# Eigenfraud — EDA & Pipeline Sanity Check

Verify the data pipeline before training:
1. Confirm images load and transforms produce no NaNs
2. Plot 2D log-power spectra (real vs fake)
3. Plot 1D radial profiles stacked
4. Check label balance

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.transforms import (
    to_grayscale_array,
    log_power_spectrum_2d,
    azimuthal_average_fast,
    spectral_residual,
    compute_mean_spectrum,
)
from src.dataset import FrequencyDataset, make_splits

# ---- EDIT THIS PATH ONCE CIFAKE IS DOWNLOADED ----
DATA_ROOT = '../data/processed/cifake'
# Expected layout: cifake/real/*.jpg, cifake/fake/*.jpg

## 1. Dataset loading & label balance

In [ ]:
dataset = FrequencyDataset(root=DATA_ROOT, size=224)
counts = dataset.label_counts()
print(f'Total samples: {len(dataset)}')
print(f'Label counts : {counts}')
assert counts['real'] > 0 and counts['fake'] > 0, 'Missing class!'

## 2. Single-image pipeline check — no NaNs, sane ranges

In [ ]:
spec2d, prof1d, label = dataset[0]

print(f'spectrum_2d shape : {spec2d.shape}  dtype={spec2d.dtype}')
print(f'  min={spec2d.min():.4f}  max={spec2d.max():.4f}  NaN={spec2d.isnan().any()}')
print(f'profile_1d shape  : {prof1d.shape}  dtype={prof1d.dtype}')
print(f'  min={prof1d.min():.4f}  max={prof1d.max():.4f}  NaN={prof1d.isnan().any()}')
print(f'label             : {label}  (0=real, 1=fake)')

assert not spec2d.isnan().any()
assert not prof1d.isnan().any()
print('✓ No NaNs')

## 3. Plot 2D log-power spectra: real vs fake examples

In [ ]:
# Find one real and one fake sample
real_idx = next(i for i, (_, lbl) in enumerate(dataset.samples) if lbl == 0)
fake_idx = next(i for i, (_, lbl) in enumerate(dataset.samples) if lbl == 1)

spec_real, _, _ = dataset[real_idx]
spec_fake, _, _ = dataset[fake_idx]

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, spec, title in zip(axes,
                            [spec_real[0].numpy(), spec_fake[0].numpy()],
                            ['Real', 'Fake (CIFAKE)']):
    im = ax.imshow(spec, cmap='inferno', origin='lower')
    ax.set_title(title)
    ax.set_xlabel('u (frequency)')
    ax.set_ylabel('v (frequency)')
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Log-Power Spectrum — centered (DC at center)')
plt.tight_layout()
plt.savefig('../figures/sanity_spectra.png', dpi=150)
plt.show()

## 4. Mean spectra & spectral residual (Figure 1 prototype)

In [ ]:
import random
random.seed(42)
N = 200  # images per class for the mean

real_paths = [p for p, l in dataset.samples if l == 0]
fake_paths = [p for p, l in dataset.samples if l == 1]

real_sample = random.sample(real_paths, min(N, len(real_paths)))
fake_sample = random.sample(fake_paths, min(N, len(fake_paths)))

print(f'Computing mean spectra over {len(real_sample)} real + {len(fake_sample)} fake images...')
mean_real = compute_mean_spectrum(real_sample)
mean_fake = compute_mean_spectrum(fake_sample)
residual  = spectral_residual(mean_fake, mean_real)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
titles = ['Mean Real Spectrum', 'Mean Fake Spectrum', 'Residual (Fake − Real)']
maps   = ['inferno', 'inferno', 'RdBu_r']
for ax, data, title, cmap in zip(axes,
                                  [mean_real, mean_fake, residual],
                                  titles, maps):
    vmax = np.abs(data).max()
    kw = dict(vmin=-vmax, vmax=vmax) if 'Residual' in title else {}
    im = ax.imshow(data, cmap=cmap, origin='lower', **kw)
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle('Figure 1 prototype — CIFAKE')
plt.tight_layout()
plt.savefig('../figures/fig1_prototype.png', dpi=150)
plt.show()

## 5. Radial profiles (Figure 2 prototype)

In [ ]:
# Compute mean radial profiles
real_profiles = []
fake_profiles = []

for path in real_sample:
    gray = to_grayscale_array(Image.open(path))
    spec = log_power_spectrum_2d(gray)
    real_profiles.append(azimuthal_average_fast(spec))

for path in fake_sample:
    gray = to_grayscale_array(Image.open(path))
    spec = log_power_spectrum_2d(gray)
    fake_profiles.append(azimuthal_average_fast(spec))

mean_real_profile = np.mean(real_profiles, axis=0)
mean_fake_profile = np.mean(fake_profiles, axis=0)
r = np.arange(len(mean_real_profile))

plt.figure(figsize=(9, 4))
plt.plot(r, mean_real_profile, label='Real', color='steelblue', linewidth=1.5)
plt.plot(r, mean_fake_profile, label='Fake (CIFAKE)', color='tomato', linewidth=1.5)
plt.xlabel('Radial frequency $r$')
plt.ylabel('Mean log-power')
plt.title('Figure 2 prototype — Radial Profiles')
plt.legend()
plt.tight_layout()
plt.savefig('../figures/fig2_prototype.png', dpi=150)
plt.show()

print(f'Profile length (r_max): {len(mean_real_profile)}')

## 6. DataLoader stress test

In [ ]:
from torch.utils.data import DataLoader

train_ds, val_ds, test_ds = make_splits(dataset)
print(f'Split sizes — train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}')

loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
spec2d_batch, prof1d_batch, labels_batch = next(iter(loader))

print(f'Batch spec2d : {spec2d_batch.shape}')
print(f'Batch prof1d : {prof1d_batch.shape}')
print(f'Batch labels : {labels_batch.shape}  unique={labels_batch.unique().tolist()}')
assert not spec2d_batch.isnan().any()
assert not prof1d_batch.isnan().any()
print('✓ DataLoader OK')